# Model Comparison — AI4I 2020 Predictive Maintenance

## Objective

Compare leakage-safe machine-learning models using:
1. the raw feature set
2. the engineered feature set

The target is `Machine failure`.

Because failures represent only a small minority of observations, the primary
evaluation metrics are Precision, Recall, F1, ROC-AUC and Average Precision.

The final holdout is **not** used in this notebook.

In [23]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../data/raw/ai4i2020.csv")
df = pd.read_csv(DATA_PATH)

TARGET = "Machine failure"
RANDOM_STATE = 42

raw_features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

engineered_features = raw_features + [
    "temp_diff",
    "power_w",
    "wear_torque",
    "strain_ratio"
]

# Recreate the engineered features so this notebook is independently executable.
df["temp_diff"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power_w"] = (
    df["Torque [Nm]"] * df["Rotational speed [rpm]"] * 2 * np.pi / 60
)
strain_limits = {"L": 11000, "M": 12000, "H": 13000}
df["wear_torque"] = df["Tool wear [min]"] * df["Torque [Nm]"]
df["strain_ratio"] = df["wear_torque"] / df["Type"].map(strain_limits)

print("Shape:", df.shape)

Shape: (10000, 18)


## 1. Development / final holdout split

To keep the final evaluation untouched, use the first 80% of the original ordered
records as development data and the final 20% as a holdout.

This ordering is used deliberately for the final stress test; it is **not** treated
as a real timestamp.

In [24]:
split_point = int(len(df) * 0.8)

dev = df.iloc[:split_point].copy()
holdout = df.iloc[split_point:].copy()

print("Development shape:", dev.shape)
print("Holdout shape:", holdout.shape)
print("Development failures:", int(dev[TARGET].sum()))
print("Holdout failures:", int(holdout[TARGET].sum()))

Development shape: (8000, 18)
Holdout shape: (2000, 18)
Development failures: 300
Holdout failures: 39


## 2. Leakage checks

Failure-mode indicators and identifiers are excluded explicitly.

In [25]:
forbidden = {
    "UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF", TARGET
}

for feature_set in [raw_features, engineered_features]:
    leaked = forbidden.intersection(feature_set)
    print("Feature set:", "raw" if feature_set == raw_features else "engineered")
    print("Forbidden columns:", leaked)
    assert not leaked

Feature set: raw
Forbidden columns: set()
Feature set: engineered
Forbidden columns: set()


## 3. Preprocessing

`Type` is one-hot encoded rather than represented as an arbitrary numeric scale.

The same preprocessing is fitted inside each CV fold to avoid information leakage.

In [26]:
def make_preprocessor(feature_columns):
    categorical = ["Type"]
    numeric = [c for c in feature_columns if c != "Type"]

    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
            ("num", "passthrough", numeric),
        ],
        remainder="drop"
    )

from sklearn.compose import ColumnTransformer

## 4. Model definitions

The model ladder starts with a trivial baseline and then increasingly flexible models.

In [27]:
def make_models():
    return {
        "Dummy": DummyClassifier(strategy="prior"),
        "LogReg": Pipeline([
            ("prep", make_preprocessor(current_features)),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),
        "RandomForest": Pipeline([
            ("prep", make_preprocessor(current_features)),
            ("model", RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced_subsample",
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ]),
        "HistGB": Pipeline([
            ("prep", make_preprocessor(current_features)),
            ("model", HistGradientBoostingClassifier(
                max_iter=300,
                learning_rate=0.05,
                random_state=RANDOM_STATE
            ))
        ])
    }

## 5. Optional XGBoost

XGBoost is included only when it is already installed. Its use is not required for
the notebook to run.

In [28]:
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

print("XGBoost available:", XGBOOST_AVAILABLE)

XGBoost available: True


In [29]:
def add_xgboost(models):
    if not XGBOOST_AVAILABLE:
        return models

    # XGBoost receives one-hot encoded Type via a preprocessing pipeline.
    models["XGBoost"] = Pipeline([
        ("prep", make_preprocessor(current_features)),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            scale_pos_weight=1.0
        ))
    ])
    return models

## 6. Cross-validation evaluation

Use stratified 5-fold CV on the development data.

Average Precision is reported instead of treating accuracy as the primary metric.

In [30]:
def evaluate_model_cv(model, X, y, cv):
    rows = []
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        m = clone(model)
        m.fit(X.iloc[train_idx], y.iloc[train_idx])

        score = m.predict_proba(X.iloc[test_idx])[:, 1]
        pred = (score >= 0.5).astype(int)

        rows.append({
            "fold": fold,
            "precision": precision_score(y.iloc[test_idx], pred, zero_division=0),
            "recall": recall_score(y.iloc[test_idx], pred, zero_division=0),
            "f1": f1_score(y.iloc[test_idx], pred, zero_division=0),
            "roc_auc": roc_auc_score(y.iloc[test_idx], score),
            "average_precision": average_precision_score(y.iloc[test_idx], score),
        })

    return pd.DataFrame(rows)

In [31]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []
fold_results = []

y_dev = dev[TARGET].astype(int).reset_index(drop=True)

for feature_name, feature_columns in [
    ("raw", raw_features),
    ("engineered", engineered_features)
]:
    current_features = feature_columns

    models = make_models()
    models = add_xgboost(models)

    X_dev = dev[feature_columns].reset_index(drop=True)

    for model_name, model in models.items():
        fold_df = evaluate_model_cv(model, X_dev, y_dev, cv)
        fold_df["model"] = model_name
        fold_df["features"] = feature_name
        fold_results.append(fold_df)

        results.append({
            "model": model_name,
            "features": feature_name,
            "precision_mean": fold_df["precision"].mean(),
            "recall_mean": fold_df["recall"].mean(),
            "f1_mean": fold_df["f1"].mean(),
            "roc_auc_mean": fold_df["roc_auc"].mean(),
            "average_precision_mean": fold_df["average_precision"].mean(),
            "precision_std": fold_df["precision"].std(ddof=1),
            "recall_std": fold_df["recall"].std(ddof=1),
            "f1_std": fold_df["f1"].std(ddof=1),
            "roc_auc_std": fold_df["roc_auc"].std(ddof=1),
            "average_precision_std": fold_df["average_precision"].std(ddof=1),
        })

results_df = pd.DataFrame(results).sort_values(
    "average_precision_mean",
    ascending=False
)

fold_results_df = pd.concat(fold_results, ignore_index=True)

results_df

,model,features,precision_mean,recall_mean,f1_mean,roc_auc_mean,average_precision_mean,precision_std,recall_std,f1_std,roc_auc_std,average_precision_std
7,RandomForest,engineered,0.944172,0.843333,0.890009,0.978571,0.906620,0.023037,0.060782,0.037631,0.012160,0.034961
9,XGBoost,engineered,0.932015,0.810000,0.866435,0.980338,0.903334,0.077644,0.065192,0.068395,0.012342,0.050266
8,HistGB,engineered,0.949731,0.823333,0.881867,0.976846,0.902181,0.041002,0.048016,0.043837,0.014374,0.044783
3,HistGB,raw,0.873038,0.656667,0.749095,0.973807,0.830168,0.098021,0.072265,0.080268,0.014520,0.076832
4,XGBoost,raw,0.874198,0.646667,0.743126,0.977636,0.825805,0.042199,0.038006,0.036870,0.012183,0.065111
2,RandomForest,raw,0.815479,0.590000,0.683863,0.970733,0.767257,0.067421,0.068313,0.066125,0.013609,0.063084
6,LogReg,engineered,0.185370,0.843333,0.303922,0.925879,0.465232,0.005463,0.025276,0.008738,0.008135,0.045512
1,LogReg,raw,0.147853,0.796667,0.249369,0.894970,0.439695,0.004754,0.036132,0.007717,0.009957,0.057571
0,Dummy,raw,0.000000,0.000000,0.000000,0.500000,0.037500,0.000000,0.000000,0.000000,0.000000,0.000000
5,Dummy,engineered,0.000000,0.000000,0.000000,0.500000,0.037500,0.000000,0.000000,0.000000,0.000000,0.000000


## 7. Compare raw vs engineered features

Look for whether engineering consistently improves the models rather than whether a
single metric improves by chance.

In [32]:
pivot = results_df.pivot(
    index="model",
    columns="features",
    values=["f1_mean", "average_precision_mean", "roc_auc_mean"]
)

pivot

f1_mean           average_precision_mean            \
features     engineered       raw             engineered       raw   
model                                                                
Dummy          0.000000  0.000000               0.037500  0.037500   
HistGB         0.881867  0.749095               0.902181  0.830168   
LogReg         0.303922  0.249369               0.465232  0.439695   
RandomForest   0.890009  0.683863               0.906620  0.767257   
XGBoost        0.866435  0.743126               0.903334  0.825805   

             roc_auc_mean            
features       engineered       raw  
model                                
Dummy            0.500000  0.500000  
HistGB           0.976846  0.973807  
LogReg           0.925879  0.894970  
RandomForest     0.978571  0.970733  
XGBoost          0.980338  0.977636

## 8. Confusion matrices for the strongest candidates

These use out-of-fold predictions on the development data.

In [33]:
oof_predictions = []

for feature_name, feature_columns in [
    ("raw", raw_features),
    ("engineered", engineered_features)
]:
    current_features = feature_columns
    models = add_xgboost(make_models())

    X_dev = dev[feature_columns].reset_index(drop=True)

    for model_name, model in models.items():
        oof = np.zeros(len(y_dev))

        for train_idx, test_idx in cv.split(X_dev, y_dev):
            m = clone(model)
            m.fit(X_dev.iloc[train_idx], y_dev.iloc[train_idx])
            oof[test_idx] = m.predict_proba(X_dev.iloc[test_idx])[:, 1]

        best = results_df[
            (results_df["model"] == model_name) &
            (results_df["features"] == feature_name)
        ].iloc[0]

        oof_predictions.append({
            "model": model_name,
            "features": feature_name,
            "scores": oof
        })

        cm = confusion_matrix(
            y_dev,
            (oof >= 0.5).astype(int),
            labels=[0, 1]
        )

        print(
            f"{model_name} / {feature_name}\n"
            f"TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}\n"
        )

Dummy / raw
TN=7700, FP=0, FN=300, TP=0

LogReg / raw
TN=6322, FP=1378, FN=61, TP=239

RandomForest / raw
TN=7660, FP=40, FN=123, TP=177

HistGB / raw
TN=7671, FP=29, FN=103, TP=197

XGBoost / raw
TN=7672, FP=28, FN=106, TP=194

Dummy / engineered
TN=7700, FP=0, FN=300, TP=0

LogReg / engineered
TN=6588, FP=1112, FN=47, TP=253

RandomForest / engineered
TN=7685, FP=15, FN=47, TP=253

HistGB / engineered
TN=7687, FP=13, FN=53, TP=247

XGBoost / engineered
TN=7682, FP=18, FN=57, TP=243



## Conclusion

This notebook establishes the leakage-safe ML benchmark.

Do not declare a final model from this notebook alone. The next stage tests whether
the reported performance is robust under a different validation strategy.